# Evaluating Deep Learning vs. Traditional ML for Phishing Detection in Resource-Constrained African Environments
This notebook implements a scientifically rigorous methodology for detecting phishing URLs by testing global phishing threats against African infrastructure, using both deep learning (BERT) and lightweight (XGBoost) models.

**Key Features of this Methodology:**
1. **African Context Dataset:** Collects genuine legitimate domains from Sub-Saharan Africa.
2. **Global Phishing Threats:** Uses OpenPhish and PhishTank (intentionally excluding URLhaus to avoid malware contamination) to represent global threats against African networks.
3. **Domain-Level Splitting:** Eliminates data leakage by strictly separating domains between training and testing sets.
4. **Model Comparison & Deployment Analysis:** Compares BERT against XGBoost, specifically analyzing inference speeds to evaluate deployment feasibility in resource-constrained environments.


In [ ]:
!pip install transformers==4.30.0 -q
!pip install datasets -q
!pip install scikit-learn -q
!pip install xgboost -q
!pip install tldextract -q

import torch
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import re
import tldextract
from urllib.parse import urlparse
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")


## 1. True African Legitimate Dataset Collection
We download the Tranco Top 1 Million sites and filter them specifically for Sub-Saharan African TLDs to build a genuine regional dataset.


In [ ]:
african_tlds = [
    'rw', 'ke', 'za', 'ng', 'gh', 'ug', 'tz', 'zm', 'zw', 'mw', 
    'mz', 'bw', 'na', 'ls', 'sz', 'sn', 'ci', 'cm', 'mg', 'mu'
]

print("Downloading Tranco Top 1M list...")
response = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=60)
african_legit_urls = []

if response.status_code == 200:
    z = zipfile.ZipFile(io.BytesIO(response.content))
    csv_filename = z.namelist()[0]
    
    with z.open(csv_filename) as f:
        tranco_df = pd.read_csv(f, names=['rank', 'domain'])
        
    print(f"Total domains downloaded: {len(tranco_df)}")
    
    # Filter for African domains
    def is_african_domain(domain):
        ext = tldextract.extract(str(domain))
        return ext.suffix.split('.')[-1] in african_tlds
    
    # Apply filter
    african_domains = tranco_df[tranco_df['domain'].apply(is_african_domain)]['domain'].tolist()
    african_legit_urls = ['https://' + domain for domain in african_domains]
    
    print(f"✅ Extracted {len(african_legit_urls)} genuine Sub-Saharan African legitimate domains!")
    
    # Let's see some examples
    print("\nSample African domains found:")
    for url in african_legit_urls[:10]:
        print("  -", url)
else:
    print("Failed to download Tranco list.")

legit_df = pd.DataFrame({'url': african_legit_urls, 'label': 0})


## 2. Global Phishing Dataset Collection
We collect active malicious URLs from OpenPhish and PhishTank.


In [ ]:
print("Collecting phishing URLs...")
phishing_urls = []

# OpenPhish
try:
    response = requests.get("https://openphish.com/feed.txt", timeout=30)
    if response.status_code == 200:
        urls = [u.strip() for u in response.text.strip().split('\n') if u.strip()]
        phishing_urls.extend(urls)
        print(f"✅ OpenPhish: {len(urls)} URLs")
except Exception as e:
    print(f"⚠️ OpenPhish Error: {e}")

# PhishTank Mirror
try:
    response = requests.get("https://raw.githubusercontent.com/mitchellkrogza/Phishing.Database/master/phishing-links-ACTIVE.txt", timeout=30)
    if response.status_code == 200:
        urls = [u.strip() for u in response.text.strip().split('\n') if u.strip() and u.startswith('http')]
        phishing_urls.extend(urls)
        print(f"✅ PhishTank mirror: {len(urls)} URLs")
except Exception as e:
    print(f"⚠️ PhishTank Error: {e}")

phishing_urls = list(set(phishing_urls))
print(f"Total unique phishing URLs: {len(phishing_urls)}")

phish_df = pd.DataFrame({'url': phishing_urls, 'label': 1})


## 3. Dataset Balancing and Domain Extraction
We combine the datasets, clean them, and crucially, extract the **Root Domain** to prevent data leakage.


In [ ]:
full_df = pd.concat([legit_df, phish_df], ignore_index=True)
full_df = full_df.drop_duplicates(subset=['url']).dropna()

# Extract Root Domain
def extract_root_domain(url):
    ext = tldextract.extract(url)
    return f"{ext.domain}.{ext.suffix}"

full_df['root_domain'] = full_df['url'].apply(extract_root_domain)

# Balance the dataset to the size of the smaller class
min_class_size = min(len(full_df[full_df['label'] == 0]), len(full_df[full_df['label'] == 1]))
print(f"Balancing dataset to {min_class_size} samples per class...")

balanced_df = pd.concat([
    full_df[full_df['label'] == 0].sample(n=min_class_size, random_state=42),
    full_df[full_df['label'] == 1].sample(n=min_class_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Balanced dataset size: {len(balanced_df)} URLs")
print(f"Unique Root Domains: {balanced_df['root_domain'].nunique()}")


## 4. Domain-Level Splitting (Crucial to Fix Data Leakage)
Instead of a random split, we use `GroupShuffleSplit` so that all URLs sharing the same root domain go entirely into either train, val, or test. This forces the models to generalize.


In [ ]:
gss_train_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# First split: Train + Val vs Test
train_val_idx, test_idx = next(gss_train_test.split(balanced_df, groups=balanced_df['root_domain']))
train_val_df = balanced_df.iloc[train_val_idx]
test_df = balanced_df.iloc[test_idx].copy()

# Second split: Train vs Val (from train_val_df)
gss_train_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss_train_val.split(train_val_df, groups=train_val_df['root_domain']))
train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Verify NO LEAKAGE
train_domains = set(train_df['root_domain'])
test_domains = set(test_df['root_domain'])
leakage = train_domains.intersection(test_domains)

print(f"\nLeakage Check: {len(leakage)} domains overlap between Train and Test.")
if len(leakage) == 0:
    print("✅ PERFECT! Zero data leakage.")
else:
    print("⚠️ WARNING: Data leakage detected!")


## 5. Lightweight Model: XGBoost
We extract TF-IDF character n-gram features from the URLs and train a fast XGBoost model.


In [ ]:
print("Extracting TF-IDF Character features...")
# Character n-grams work well for URL structure (e.g. '.php?id=', 'login')
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=5000)

X_train = vectorizer.fit_transform(train_df['url'])
X_test = vectorizer.transform(test_df['url'])

y_train = train_df['label']
y_test = test_df['label']

print("Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=200, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist' # Fast training
)

xgb_model.fit(X_train, y_train)

# Evaluate
import time
start_time = time.time()
xgb_preds = xgb_model.predict(X_test)
xgb_infer_time = time.time() - start_time
xgb_acc = accuracy_score(y_test, xgb_preds)

print(f"\n{'='*40}")
print(f"🏆 XGBoost Test Accuracy: {xgb_acc*100:.2f}%")
print(f"⏱️ Inference Time for {X_test.shape[0]} URLs: {xgb_infer_time:.4f} seconds")
print(f"⚡ Time per 1000 URLs: {(xgb_infer_time / X_test.shape[0]) * 1000:.4f} seconds")
print(f"{'='*40}\n")
print(classification_report(y_test, xgb_preds, target_names=['Legitimate', 'Phishing']))


## 6. Deep Learning Model: BERT
We train BERT on the same leak-free dataset. Due to the strict domain splitting, accuracy will be realistic (likely 85-92%) rather than the artificial 99.9%.


In [ ]:
class URLDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        url = str(self.data.iloc[idx]['url'])
        label = int(self.data.iloc[idx]['label'])

        encoding = self.tokenizer(
            url,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
bert_model.to(device)

train_dataset = URLDataset(train_df, tokenizer)
test_dataset = URLDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

optimizer = AdamW(bert_model.parameters(), lr=2e-5)

print("Training BERT for 1 epoch...")
bert_model.train()
for batch in tqdm(train_loader, desc="Training BERT"):
    optimizer.zero_grad()
    
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['label'].to(device)
    
    outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

print("\nEvaluating BERT...")
bert_model.eval()
bert_preds = []
bert_labels = []

import time
start_time = time.time()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing BERT"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        
        bert_preds.extend(preds.cpu().numpy())
        bert_labels.extend(labels.cpu().numpy())

bert_infer_time = time.time() - start_time
bert_acc = accuracy_score(bert_labels, bert_preds)
print(f"\n{'='*40}")
print(f"🏆 BERT Test Accuracy: {bert_acc*100:.2f}%")
print(f"⏱️ Inference Time for {len(test_df)} URLs: {bert_infer_time:.4f} seconds")
print(f"⚡ Time per 1000 URLs: {(bert_infer_time / len(test_df)) * 1000:.4f} seconds")
print(f"{'='*40}\n")
print(classification_report(bert_labels, bert_preds, target_names=['Legitimate', 'Phishing']))


## 7. Conclusions for the Paper
If XGBoost performs comparably to BERT (e.g., within 2-3% accuracy), you can state that:
> "While deep learning models like BERT offer marginal improvements, lightweight gradient boosting frameworks provide highly competitive performance with significantly lower computational overhead, making them more suitable for scalable deployment in resource-constrained African network infrastructures."
This gives your paper a very strong, novel, and publishable conclusion!
